# 多表合并

把多个数据框合成一个，是预处理的核心。Polars 有几种用途不同的合并方式。这一章讲 `concat`、`join`、`join_asof`、`join_where`，以及就地追加行、列的几个方法。

In [1]:
import numpy as np
import polars as pl
from helper.jupyter import row

## concat

`pl.concat()` 把多个数据框纵向或横向合并，方式由 `how` 指定。

- `vertical` / `vertical_relaxed` — 纵向合并（`relaxed` 会把类型提升到上位类型）
- `horizontal` — 横向合并（要求行数相同）
- `horizontal_extend` — 横向合并，行数不够的用 NULL 补
- `diagonal` / `diagonal_relaxed` — 列只有部分重叠时，两个方向一起合并
- `align` — 先按共同的键列把行对齐再合并

下面用两个数据框把每种方式过一遍。

In [2]:
df1 = pl.DataFrame({"x":[1, 2, 3], "y":[2, 3, 1]})
df2 = pl.DataFrame({"x":[6, 2, 1, 5], "y":[12, 3, 2, 4]})
row(df1, df2)

x,y
i64,i64
1,2
2,3
3,1
x,y
i64,i64
6,12
2,3
1,2
5,4


### 縦結合

列名和类型都一致时，直接用 `vertical` 纵向堆起来。

In [3]:
pl.concat([df1, df2], how='vertical')

x,y
i64,i64
1,2
2,3
3,1
6,12
2,3
1,2
5,4


列名相同、类型不同时用 `vertical_relaxed`，结果会统一到上位类型。下面的代码先把 `df2` 的 `x` 转成 `Float64` 再合并，所以 `x` 是 `Float64`。

In [4]:
pl.concat([
    df1, 
    df2.with_columns(pl.col('x').cast(pl.Float64))
    ], 
    how='vertical_relaxed')

x,y
f64,i64
1.0,2
2.0,3
3.0,1
6.0,12
2.0,3
1.0,2
5.0,4


想区分行来自哪个数据框，就在合并前加一列标记。下面的代码给 `df1` 和 `df2` 各加一个 `key` 列再纵向堆起来。

In [5]:
data = {"A":df1, "B":df2}
pl.concat([
    df.select(pl.lit(key).alias("key"), pl.all()) 
    for key, df in data.items()
])

key,x,y
str,i64,i64
"""A""",1,2
"""A""",2,3
"""A""",3,1
"""B""",6,12
"""B""",2,3
"""B""",1,2
"""B""",5,4


### 横結合

列名不同的两个数据框横向并排，用 `horizontal`。它要求行数一致，所以这里给列名加后缀来区分。

Polars 2.0 里 `horizontal` 遇到行数不同会直接报错。想用 NULL 补齐短的那一边，就指定 `horizontal_extend`。

In [6]:
# 列名分别加 1 / 2 后并排（df2 多一行，所以用 horizontal_extend）
pl.concat(
    [
        df1.rename(lambda name: f"{name}1"),
        df2.select(pl.all().name.suffix("2")),
    ],
    how="horizontal_extend",
)

x1,y1,x2,y2
i64,i64,i64,i64
1,2,6,12
2,3,2,3
3,1,1,2
null,null,5,4


只有部分列名重叠时用 `diagonal` 或 `diagonal_relaxed`。`diagonal_relaxed` 会把类型提升到上位类型。下面的代码给 `df1` 加 `u`、给 `df2` 加 `v`，重叠的列纵向对齐，不重叠的列用 NULL 填。

### 纵向与横向合并

In [7]:
dfs = [
    df1.with_columns(u=pl.col('x') + pl.col('y')),
    df2.with_columns(v=pl.col('x') * pl.col('y'))
]
pl.concat(dfs, how='diagonal')

x,y,u,v
i64,i64,i64,i64
1,2,3,null
2,3,5,null
3,1,4,null
6,12,null,72
2,3,null,6
1,2,null,2
5,4,null,20


### 整列結合

`align` 和 `diagonal` 类似，但它先按共同的列把行对齐再合并。下面的例子用 `x` 和 `y` 作键，所以 `x=1, y=2` 那一行的 `u=3` 和 `v=2` 会并成一行。

In [8]:
pl.concat(dfs, how="align")

x,y,u,v
i64,i64,i64,i64
1,2,3,2
2,3,5,6
3,1,4,null
5,4,null,20
6,12,null,72


用 `pl.align_frames()` 可以按指定列把多个数据框对齐。对齐之后，各数据框的行数和键的顺序都一致。

In [9]:
row(*pl.align_frames(*dfs, on=['x', 'y']))

x,y,u
i64,i64,i64
1,2,3
2,3,5
3,1,4
5,4,null
6,12,null
x,y,v
i64,i64,i64
1,2,2
2,3,6


## join

`join()` 按键把两个数据框对上。主要参数如下。

```python
df.join(
    other,                     # 要连接的另一张表
    on=None,                   # 两张表共同的键列（或表达式）
    how="inner",               # 连接方式
    left_on=None,              # 只有左表用的键
    right_on=None,             # 只有右表用的键
    suffix="_right",           # 重名列的后缀
    validate="m:m",            # 校验键的对应关系（"m:m" / "m:1" / "1:m" / "1:1"）
    nulls_equal=False,         # NULL 之间算不算匹配（2.0 由 join_nulls 改名）
    coalesce=None,             # 是否把键列合成一列
    maintain_order=None,       # 是否保持行序（"left" / "right" / "none" 等）
)
```

`how` 的取值：

- **inner** — 只返回两边都匹配上的行
- **left** — 左表全部行，加上右表匹配上的行
- **right** — 右表全部行，加上左表匹配上的行
- **full** — 两边任意一边匹配上的行全要
- **semi** — 左表中右表匹配上的行，只保留左表的列
- **anti** — 左表中右表匹配不上的行

Polars 2.0 的 `collect()` 默认走流式引擎，所以 `join` 的行序不保证。顺序有含义时请指定 `maintain_order`，或者最后 `sort()`。

In [10]:
df_left = pl.DataFrame({
    "id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "David"]
})

df_right = pl.DataFrame({
    "id": [3, 4, 5],
    "age": [23, 30, 40]
})

### inner

In [11]:
row(
    df_left, df_right, 
    df_left.join(df_right, on="id", how="inner")    
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


### left 与 right

`left` 保留左表所有行。下面的例子里 `id` 为 1 和 2 的行在右表没有对应，所以 `age` 是 NULL。`right` 正好相反。

In [12]:
row(
    df_left, df_right,
    df_left.join(df_right, on="id", how="left")
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


In [13]:
row(
    df_left, df_right,
    df_left.join(df_right, on="id", how="right")
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


### full

`full` 返回两边的所有行。为了区分行来自哪边，键会变成两列（`id` 和 `id_right`）。可以看出 `id` 为 1、2 只在左边，5 只在右边。

In [14]:
row(
    df_left, df_right,
    df_left.join(df_right, on="id", how="full")
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


指定 `coalesce=True` 后，这两个键列会合成一列。

In [15]:
row(
    df_left, df_right,
    df_left.join(df_right, on="id", how="full", coalesce=True)
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


### semi 与 anti

`semi` 只返回右表存在的行，`anti` 只返回右表不存在的行。两者都不包含右表的列。

In [16]:
row(
    df_left, df_right,
    df_left.join(df_right, on="id", how="semi")
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


In [17]:
row(
    df_left, df_right,
    df_left.join(df_right, on="id", how="anti")
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


### cross

`cross` 返回两个数据框的笛卡尔积：左表每行和右表每行的所有组合，所以不用指定 `on`。

In [18]:
row(
    df_left, df_right,
    df_left.join(df_right, how="cross")
)

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""
4,"""David"""
id,age
i64,i64
3,23
4,30
5,40


### validate

连接键的对应关系分四类。

* **1:1** — 两边键都唯一（例：员工 ID 与工牌号）
* **1:m** — 左边唯一、右边可重复（例：部门与所属员工）
* **m:1** — 左边可重复、右边唯一（例：订单明细与商品主数据）
* **m:m** — 两边都允许重复（默认）

有多行匹配时，默认会把**所有组合（笛卡尔积）**放进结果。指定 `validate` 就能强制预期的关系，违反时报错。

下面两个数据框里，`id == 3` 的行各有 2 条。

In [19]:
df_left = pl.DataFrame({
    "id": [1, 2, 3, 4, 3],
    "name": ["Alice", "Bob", "Charlie", "David", "charlie"]
})

df_right = pl.DataFrame({
    "id": [3, 4, 5, 3],
    "age": [23, 30, 40, 46]
})

直接用 `id` 连接时，`id == 3` 的行会变成 2 × 2 = 4 行。

In [20]:
df_left.join(df_right, on="id")

id,name,age
i64,str,i64
3,"""Charlie""",23
3,"""Charlie""",46
4,"""David""",30
3,"""charlie""",23
3,"""charlie""",46


指定 `validate="1:1"` 后，键不是一对一就会报错，这样能发现数据重复。

In [21]:
%%capture_except
df_left.join(df_right, on="id", validate='1:1')

ComputeError: join keys did not fulfill 1:1 validation


## update

`DataFrame.update()` 用另一张表的值覆盖键相同的行。下面的代码把 `id` 相同的行的 `x`、`y` 换成 `df_update` 里的值。

In [22]:
df = pl.DataFrame({
    "id": [1, 2, 3, 4],
    "x": [10, 20, 30, 40],
    "y": [40, 30, 20, 10]
})

df_update = pl.DataFrame({
    "id": [1, 3],
    "x": [100, 300],
    "y": [-10, -20],
})

df_res = df.update(df_update, on='id')
row(df, df_update, df_res)

id,x,y
i64,i64,i64
1,10,40
2,20,30
3,30,20
4,40,10
id,x,y
i64,i64,i64
1,100,-10
3,300,-20
id,x,y


## join_asof

`join_asof` 用时间或数值这类连续值做「就近匹配」。它不要求完全相等，而是找最接近的行，时序预处理里很常见。

- 一边是某个时点的快照，另一边要取最近时点的值
- 需要指定「往前找」还是「往后找」

In [23]:
import polars as pl

df1 = pl.DataFrame(
    {
        "time": [1, 5, 10, 15, 20],
        "event": ["A", "B", "C", "D", "E"],
    }
)

df2 = pl.DataFrame(
    {
        "time": [2, 6, 12, 18],
        "price": [100, 105, 110, 115],
    }
)

result = df1.join_asof(df2, on="time", strategy="backward")
row(df1, df2, result)

time,event
i64,str
1,"""A"""
5,"""B"""
10,"""C"""
15,"""D"""
20,"""E"""
time,price
i64,i64
2,100
6,105


对 `df1` 的每一行，在 `df2` 里找 `time` 不晚于它的最近一行。`strategy="backward"` 就是这个方向，改成 `"forward"` 则选未来的值。

## join_where

`join_where()` 可以把列之间的比较条件直接当连接条件。下面的例子里，`df1` 的 `time` 落在 `df2` 的 `time_span` 区间内就连接。给多个条件时，要全部满足才保留。

In [24]:
df1 = pl.DataFrame(
    {
        "id": [100, 101, 102],
        "time": [120, 140, 160],
    }
)
df2 = pl.DataFrame(
    {
        "t_id": [404, 498, 676, 742],
        "time_span": [(100, 110), (110, 130), (90, 100), (150, 170)],
    }
)
df_res = df1.join_where(
    df2,
    pl.col('time') >= pl.col('time_span').list.get(0),
    pl.col('time') <= pl.col('time_span').list.get(1)
)
row(df1, df2, df_res)

id,time
i64,i64
100,120
101,140
102,160
t_id,time_span
i64,list[i64]
404,"[100, 110]"
498,"[110, 130]"
676,"[90, 100]"
742,"[150, 170]"


布尔运算和 `is_between()` 也能用。

In [25]:
df_res1 = df1.join_where(
    df2,
    (pl.col('time') >= pl.col('time_span').list.get(0)) & (pl.col('time') <= pl.col('time_span').list.get(1))
)

df_res2 = df1.join_where(
    df2,
    pl.col('time').is_between(pl.col('time_span').list.get(0), pl.col('time_span').list.get(1))
)
row(df_res1, df_res2)

id,time,t_id,time_span
i64,i64,i64,list[i64]
100,120,498,"[110, 130]"
102,160,742,"[150, 170]"
id,time,t_id,time_span
i64,i64,i64,list[i64]
100,120,498,"[110, 130]"
102,160,742,"[150, 170]"


`join_where()` 内部先做两张表的笛卡尔积再按条件过滤，所以行数一大就非常慢。下面的例子从两个点集里找距离小于 0.01 的点对，计算量正比于 $N^2$。

In [26]:
n = 4000
np.random.seed(42)
df1 = pl.DataFrame(dict(
    x1=np.random.normal(size=n),
    y1=np.random.normal(size=n),
))
df2 = pl.DataFrame(dict(
    x2=np.random.normal(size=n),
    y2=np.random.normal(size=n),
))

In [27]:
%%time
df_res1 = (
    df1
    .join_where(
        df2, 
        (pl.col('x2') - pl.col('x1'))**2 + (pl.col('y2') - pl.col('y1'))**2 < 0.0001
    )
    .sort('x1', 'y1')
)

CPU times: total: 93.8 ms
Wall time: 23.1 ms


这种处理用 SciPy 的 `KDTree` 会快很多。`query_ball_tree()` 求出范围内成对的点，再按行号把 `df1` 和 `df2` 连起来。

In [28]:
from scipy.spatial import KDTree

In [29]:
%%time
tree1 = KDTree(df1.to_numpy())
tree2 = KDTree(df2.to_numpy())
res = tree1.query_ball_tree(tree2, 0.01)

df_pair = (
    pl.DataFrame(pl.Series('index2', res, dtype=pl.List(pl.UInt32))) # 把 res 转成数据框
    .with_row_index('index1')
    .filter(pl.col('index2').list.len() > 0) # 只保留有近距离点的行
    .explode('index2') # 展开列表，一行一对
)
df_res2 = (
    df1
    .with_row_index('index1')
    .join(df_pair, on='index1')
    .join(
        df2.with_row_index('index2'), on='index2')
    .drop('index1', 'index2')
    .sort('x1', 'y1')
)

CPU times: total: 15.6 ms
Wall time: 12.2 ms


In [30]:
row(df_res1, df_pair, df_res2)

x1,y1,x2,y2
f64,f64,f64,f64
-2.065083,-0.70744,-2.05916,-0.70436
-1.769076,0.817306,-1.768415,0.825853
-1.523187,-0.501234,-1.514414,-0.501647
-1.519346,-0.035096,-1.527461,-0.03116
-1.478522,0.932435,-1.482397,0.94037
…,…,…,…
1.66907,0.388819,1.678574,0.385821
1.696456,-0.038743,1.692313,-0.037887
1.734937,0.142991,1.737985,0.135518


`res` 是双层列表，`tree1` 的每个点对应一项，里面是与它距离在 0.01 以内的 `tree2` 点编号。下面的代码只显示前 50 个点里非空的部分。

In [31]:
for i, index2 in enumerate(res[:50]):
    if index2:
        print(i, index2)

10 [1832, 2789]
41 [1502]
44 [739]
45 [2018]


## extend 与 vstack

`DataFrame.extend()` 和 `DataFrame.vstack()` 都是往已有的表里追加另一个表的行。两者都已弃用，现在推荐 `pl.concat()`。这里只把行为差别说清楚。

- **`extend()`** — 就地修改原表（同时返回修改后的表）。
- **`vstack()`** — 用 `in_place` 选择就地修改还是返回新表。

In [32]:
df1 = pl.DataFrame({"a": [1, 2], "b": [3, 4]})
df2 = pl.DataFrame({"a": [5, 6], "b": [7, 8]})

# 用 clone() 复制原表
df_extend = df1.clone()
df_extend.extend(df2)  # extend 就地修改

df_vstack = df1.clone()
df_vstack.vstack(df2, in_place=True)  # vstack 就地修改
row(df_extend, df_vstack)

a,b
i64,i64
1,3
2,4
5,7
6,8
a,b
i64,i64
1,3
2,4
5,7


两者的区别在内存怎么组织。

`extend()` 把要追加的数据拷到原内存的末尾，合并后内存是连续的。但要重新分配，所以在循环里反复调用并不划算。

`vstack()` 保留两块内存不动，用链接把它们接起来，结果的内存是不连续的，后续计算可能变慢。以前的常见做法是循环里 `vstack()`，最后 `rechunk()` 一次。

现在请改用 `pl.concat()`，需要时最后再 `rechunk()`。

In [33]:
print(f'{df_extend.n_chunks() = }')           # extend 后的 chunk 数
print(f'{df_vstack.n_chunks() = }')          # vstack 后的 chunk 数
print(f'{df_vstack.rechunk().n_chunks() = }')  # rechunk 后的 chunk 数

df_extend.n_chunks() = 1
df_vstack.n_chunks() = 2
df_vstack.rechunk().n_chunks() = 1


## hstack

`DataFrame.hstack()` 把两个数据框并排。`in_place=True` 时直接给原表加列。它也已弃用，新代码请用 `pl.concat(..., how="horizontal")` 或 `with_columns()`。

In [34]:
df1 = pl.DataFrame({"a": [1, 2], "b": [3, 4]})
df2 = pl.DataFrame({"c": [5, 6], "d": [7, 8]})

# 不就地修改，返回新表
df3 = df1.hstack(df2)

# 就地修改
df1.hstack(df2, in_place=True)
row(df3, df1)

a,b,c,d
i64,i64,i64,i64
1,3,5,7
2,4,6,8
a,b,c,d
i64,i64,i64,i64
1,3,5,7
2,4,6,8


## merge_sorted

`merge_sorted()` 把**已经按键列排好序**的两个数据框纵向合并，返回结果仍按键有序。前提是输入确实已排序。

In [35]:
df1 = pl.DataFrame({"a": [3, 1, 5], "b": [6, 4, 2]})
df2 = pl.DataFrame({"a": [6, 2, 4], "b": [7, 5, 3]})

# 按 a 列排序后再合并
df3 = df1.sort("a").merge_sorted(df2.sort("a"), key="a")

# 按 b 列排序后再合并
df4 = df1.sort("b").merge_sorted(df2.sort("b"), key="b")

row(df3, df4)

a,b
i64,i64
1,4
2,5
3,6
4,3
5,2
6,7
a,b
i64,i64
5,2
